In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q pymupdf sentence-transformers transformers accelerate gradio pillow numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.5 MB/s eta 0:00:00


In [ ]:
!python -m pip install -q gradio

In [ ]:
from pathlib import Path

# ===== Paths =====
BASE_DIR = Path("/content/drive/MyDrive/MRAG")
PDF_PATH = BASE_DIR / "mutcd11theditionr1hl.pdf"
SECTIONS_JSON = BASE_DIR / "mutcd_sections_with_images.json"
PAGE_IMAGE_DIR = BASE_DIR / "page_images"

# Cache files (saved to Drive so later sessions start faster)
CACHE_DIR = BASE_DIR / "mmrag_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PAGE_RECORDS_JSON = CACHE_DIR / "page_records.json"
SECTION_EMB_NPY = CACHE_DIR / "section_embeddings.npy"
PAGE_EMB_NPY = CACHE_DIR / "page_embeddings.npy"

# ===== Models =====
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
VLM_MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

# ===== Retrieval =====
TOP_K_TEXT = 8
TOP_K_PAGE = 10
FINAL_TEXT_RESULTS = 4
FINAL_PAGE_RESULTS = 4

# ===== Generation =====
MAX_SECTION_TEXT_CHARS = 1800
MAX_PAGE_TEXT_CHARS = 1200
MAX_NEW_TOKENS = 320

print("PDF exists:", PDF_PATH.exists())
print("Sections JSON exists:", SECTIONS_JSON.exists())
print("Page image dir exists:", PAGE_IMAGE_DIR.exists())
print("Cache dir:", CACHE_DIR)

PDF exists: True
Sections JSON exists: True
Page image dir exists: True
Cache dir: /content/drive/MyDrive/MRAG/mmrag_cache


In [ ]:
import json

with open(SECTIONS_JSON, "r", encoding="utf-8") as f:
    raw_sections = json.load(f)

sections = []
for sec in raw_sections:
    page_num = int(sec["page_num"])
    image_path = sec.get("image_path", "")
    expected = PAGE_IMAGE_DIR / f"page_{page_num:04d}.png"

    if not image_path or not Path(image_path).exists():
        image_path = str(expected)

    sections.append({
        "section": str(sec.get("section", "")).strip(),
        "title": str(sec.get("title", "")).strip(),
        "page_num": page_num,
        "text": str(sec.get("text", "")).strip(),
        "image_path": image_path
    })

print("Loaded sections:", len(sections))
print(sections[0].keys() if sections else "No sections found")

Loaded sections: 1089
dict_keys(['section', 'title', 'page_num', 'text', 'image_path'])


In [ ]:
import fitz
import re

def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()

def extract_page_records_from_pdf(pdf_path: Path, page_image_dir: Path):
    doc = fitz.open(str(pdf_path))
    page_records = []

    figure_re = re.compile(r"(Figure\s+[A-Za-z0-9.\-]+[^.\n]*)", re.IGNORECASE)
    table_re = re.compile(r"(Table\s+[A-Za-z0-9.\-]+[^.\n]*)", re.IGNORECASE)
    sign_code_re = re.compile(r"\b([A-Z]{1,3}\d{1,2}[A-Za-z0-9\-]*)\b")

    for i, page in enumerate(doc):
        page_num = i + 1
        text = page.get_text("text") or ""
        text_norm = normalize_spaces(text)

        figures = figure_re.findall(text)
        tables = table_re.findall(text)
        sign_codes = list(sorted(set(sign_code_re.findall(text))))

        # use first 1-2 strong figure/table lines as caption-like cues
        caption_candidates = []
        for item in figures[:3]:
            caption_candidates.append(normalize_spaces(item))
        for item in tables[:3]:
            caption_candidates.append(normalize_spaces(item))

        page_records.append({
            "page_num": page_num,
            "image_path": str(page_image_dir / f"page_{page_num:04d}.png"),
            "page_text": text,
            "page_text_norm": text_norm,
            "figure_captions": caption_candidates,
            "table_titles": [normalize_spaces(x) for x in tables[:3]],
            "sign_codes": sign_codes[:50],
        })

    return page_records

page_records = extract_page_records_from_pdf(PDF_PATH, PAGE_IMAGE_DIR)
print("Page records built:", len(page_records))
print(page_records[0].keys())

Page records built: 1162
dict_keys(['page_num', 'image_path', 'page_text', 'page_text_norm', 'figure_captions', 'table_titles', 'sign_codes'])


In [ ]:
with open(PAGE_RECORDS_JSON, "w", encoding="utf-8") as f:
    json.dump(page_records, f, ensure_ascii=False, indent=2)

print("Saved page records to:", PAGE_RECORDS_JSON)

Saved page records to: /content/drive/MyDrive/MRAG/mmrag_cache/page_records.json


In [ ]:
def clean_for_match(text: str) -> str:
    text = (text or "").lower().replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text).strip()

section_retrieval_texts = []
for s in sections:
    section_retrieval_texts.append(
        f"Section {s['section']}\n"
        f"Title: {s['title']}\n"
        f"Page: {s['page_num']}\n"
        f"{s['text']}"
    )

page_retrieval_texts = []
for p in page_records:
    page_retrieval_texts.append(
        f"Page {p['page_num']}\n"
        f"Figure captions: {' | '.join(p['figure_captions'])}\n"
        f"Table titles: {' | '.join(p['table_titles'])}\n"
        f"Sign codes: {' | '.join(p['sign_codes'][:20])}\n"
        f"{p['page_text'][:4000]}"
    )

print("Section retrieval texts:", len(section_retrieval_texts))
print("Page retrieval texts:", len(page_retrieval_texts))

Section retrieval texts: 1089
Page retrieval texts: 1162


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

if SECTION_EMB_NPY.exists() and PAGE_EMB_NPY.exists():
    section_embeddings = np.load(SECTION_EMB_NPY)
    page_embeddings = np.load(PAGE_EMB_NPY)
    print("Loaded cached embeddings.")
else:
    section_embeddings = embed_model.encode(
        section_retrieval_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    page_embeddings = embed_model.encode(
        page_retrieval_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    np.save(SECTION_EMB_NPY, section_embeddings)
    np.save(PAGE_EMB_NPY, page_embeddings)
    print("Computed and saved embeddings.")

print("section_embeddings:", section_embeddings.shape)
print("page_embeddings:", page_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded cached embeddings.
section_embeddings: (1089, 384)
page_embeddings: (1162, 384)


In [ ]:
from transformers import pipeline

vlm = pipeline(
    "image-text-to-text",
    model=VLM_MODEL_NAME,
    device_map="auto"
)

print("VLM loaded.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

VLM loaded.


In [ ]:
PAGE_QUERY_RE = re.compile(r"\bpage\s+(\d{1,4})\b", re.IGNORECASE)

VISUAL_TERMS = [
    "figure", "fig", "diagram", "table", "chart", "image", "images",
    "show me", "what does this page show", "page ", "sign", "signs",
    "plaque", "plaques", "symbol", "symbols", "label", "labels",
    "caption", "captions", "plate", "plates", "drawing", "drawings"
]

def parse_explicit_page(query: str):
    m = PAGE_QUERY_RE.search(query or "")
    if m:
        return int(m.group(1))
    return None

def is_visual_query(query: str) -> bool:
    q = clean_for_match(query)
    return any(term in q for term in VISUAL_TERMS)

In [ ]:
def retrieve_text_sections(query: str, top_k: int = TOP_K_TEXT):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    scores = section_embeddings @ q_emb
    idxs = np.argsort(scores)[::-1][:top_k]

    q_norm = clean_for_match(query)
    results = []

    for idx in idxs:
        sec = sections[idx]
        score = float(scores[idx])

        title_norm = clean_for_match(sec["title"])
        text_norm = clean_for_match(sec["text"][:3000])

        if q_norm and q_norm in title_norm:
            score += 0.20
        if q_norm and q_norm in text_norm:
            score += 0.12

        results.append({
            "score": score,
            **sec
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [ ]:
def retrieve_pages(query: str, top_k: int = TOP_K_PAGE):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    scores = page_embeddings @ q_emb
    idxs = np.argsort(scores)[::-1][:top_k]

    q_norm = clean_for_match(query)
    visual = is_visual_query(query)
    results = []

    for idx in idxs:
        page = page_records[idx]
        score = float(scores[idx])

        page_text_norm = page["page_text_norm"]
        fig_text = clean_for_match(" | ".join(page["figure_captions"]))
        table_text = clean_for_match(" | ".join(page["table_titles"]))
        label_text = clean_for_match(" | ".join(page["sign_codes"]))

        # lexical boosts
        if q_norm and q_norm in fig_text:
            score += 0.30
        if q_norm and q_norm in table_text:
            score += 0.25
        if q_norm and q_norm in label_text:
            score += 0.16
        if q_norm and q_norm in page_text_norm:
            score += 0.10

        # visual-page preference
        if visual:
            if page["figure_captions"]:
                score += 0.08
            if page["table_titles"]:
                score += 0.06
            if page["sign_codes"]:
                score += 0.04

        results.append({
            "score": score,
            **page
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [ ]:
def page_record_by_num(page_num: int):
    if 1 <= page_num <= len(page_records):
        return page_records[page_num - 1]
    return None

def fuse_results(query: str):
    explicit_page = parse_explicit_page(query)
    visual = is_visual_query(query)

    text_results = retrieve_text_sections(query, top_k=TOP_K_TEXT)
    page_results = retrieve_pages(query, top_k=TOP_K_PAGE)

    final_text = text_results[:FINAL_TEXT_RESULTS]
    final_pages = []
    seen_pages = set()

    # Rule 1: explicit page override
    if explicit_page is not None:
        p = page_record_by_num(explicit_page)
        if p is not None:
            final_pages.append({
                "score": 999.0,
                **p
            })
            seen_pages.add(explicit_page)

            # bring neighbors optionally
            for delta in (-1, 1):
                npg = explicit_page + delta
                neighbor = page_record_by_num(npg)
                if neighbor is not None and npg not in seen_pages:
                    final_pages.append({
                        "score": 998.0 - abs(delta),
                        **neighbor
                    })
                    seen_pages.add(npg)

    # Rule 2: visual queries prioritize page retriever
    if visual:
        for p in page_results:
            if p["page_num"] not in seen_pages:
                final_pages.append(p)
                seen_pages.add(p["page_num"])
            if len(final_pages) >= FINAL_PAGE_RESULTS:
                break

    # Rule 3: for non-visual queries, bring section pages first
    else:
        for t in final_text:
            pg = t["page_num"]
            if pg not in seen_pages:
                p = page_record_by_num(pg)
                if p is not None:
                    final_pages.append({
                        "score": t["score"],
                        **p
                    })
                    seen_pages.add(pg)

        for p in page_results:
            if p["page_num"] not in seen_pages:
                final_pages.append(p)
                seen_pages.add(p["page_num"])
            if len(final_pages) >= FINAL_PAGE_RESULTS:
                break

    final_pages = final_pages[:FINAL_PAGE_RESULTS]
    return final_text, final_pages

In [ ]:
def split_sentences(text: str):
    raw = (text or "").replace("\n", " ").strip()
    parts = re.split(r'(?<=[.!?])\s+', raw)
    return [p.strip() for p in parts if len(p.strip()) > 20]

def fallback_answer(query: str, text_results):
    if not text_results:
        return "No relevant MUTCD sections found."

    evidence = []
    for r in text_results[:3]:
        for s in split_sentences(r["text"])[:8]:
            evidence.append((r, s))

    if not evidence:
        top = text_results[0]
        return f"Relevant evidence was retrieved from Section {top['section']}, Page {top['page_num']}, but I could not form a clean answer."

    sent_texts = [x[1] for x in evidence]
    sent_emb = embed_model.encode(
        sent_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    scores = sent_emb @ q_emb
    order = np.argsort(scores)[::-1]

    chosen = []
    seen = set()
    for idx in order:
        r, s = evidence[idx]
        key = (r["section"], s)
        if key in seen:
            continue
        seen.add(key)
        chosen.append((r, s))
        if len(chosen) >= 5:
            break

    answer = " ".join([x[1] for x in chosen])

    cites = []
    seen_cites = set()
    for r, _ in chosen:
        c = f"Section {r['section']}, Page {r['page_num']}"
        if c not in seen_cites:
            seen_cites.add(c)
            cites.append(c)

    if cites:
        answer += "\n\nCitations:\n" + "\n".join([f"- {c}" for c in cites])

    return answer

In [ ]:
def build_vlm_messages(question: str, text_results, page_results):
    content = []

    # page images
    for p in page_results[:FINAL_PAGE_RESULTS]:
        img_path = p["image_path"]
        if img_path and Path(img_path).exists():
            content.append({"type": "image", "image": Image.open(img_path)})

    text_blocks = []
    for i, r in enumerate(text_results[:FINAL_TEXT_RESULTS], 1):
        text_blocks.append(
            f"""[Text Source {i}]
Section: {r['section']}
Title: {r['title']}
Page: {r['page_num']}
Text:
{r['text'][:MAX_SECTION_TEXT_CHARS]}
"""
        )

    page_blocks = []
    for i, p in enumerate(page_results[:FINAL_PAGE_RESULTS], 1):
        page_blocks.append(
            f"""[Page Source {i}]
Page: {p['page_num']}
Figure captions: {' | '.join(p['figure_captions'])}
Table titles: {' | '.join(p['table_titles'])}
Sign codes: {' | '.join(p['sign_codes'][:20])}
"""
        )

    prompt = f"""
You are answering a question using retrieved MUTCD evidence.

Use both evidence channels equally:
- text sources for exact MUTCD wording
- page images for figures, diagrams, tables, sign sheets, captions, labels, and page layout

Rules:
1. Use only the provided text and page images.
2. If the question is about a specific page or figure, prioritize describing that page.
3. If the question is visual, explicitly describe what the relevant page(s) show.
4. If the question is textual, use the section wording precisely.
5. Write a clear answer in 3 to 7 sentences.
6. End with citations.
7. Do not invent facts.

Question:
{question}

Retrieved text evidence:
{chr(10).join(text_blocks)}

Retrieved page evidence:
{chr(10).join(page_blocks)}

Output format:

Answer:
<clear explanation>

Citations:
- Section X, Page Y
- Page Z
""".strip()

    content.append({"type": "text", "text": prompt})
    return [{"role": "user", "content": content}]

In [ ]:
def ask_vlm(question: str, text_results, page_results):
    messages = build_vlm_messages(question, text_results, page_results)

    output = vlm(
        text=messages,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False
    )

    generated = output[0]["generated_text"]
    if isinstance(generated, list):
        return generated[-1]["content"]
    return generated

In [ ]:
def answer_question(question: str):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", [], ""

    text_results, page_results = fuse_results(question)

    try:
        answer = ask_vlm(question, text_results, page_results)
        if not answer or len(answer.strip()) < 40:
            answer = fallback_answer(question, text_results)
    except Exception:
        answer = fallback_answer(question, text_results)

    gallery_items = []
    for p in page_results:
        img_path = p["image_path"]
        if img_path and Path(img_path).exists():
            caption = (
                f"Page {p['page_num']} | "
                f"Figures: {' | '.join(p['figure_captions'])} | "
                f"Tables: {' | '.join(p['table_titles'])}"
            )
            gallery_items.append((img_path, caption))

    ref_lines = []
    for r in text_results:
        ref_lines.append(f"Section {r['section']} — {r['title']} (Page {r['page_num']})")
    for p in page_results:
        ref_lines.append(f"Page {p['page_num']} — visual evidence")

    ref_text = "\n".join(dict.fromkeys(ref_lines))
    return answer, gallery_items, ref_text

In [ ]:
tests = [
    "What does MUTCD say about pedestrian hybrid beacons?",
    "Warning Signs and Plaques for Grade Crossings",
    "What does page 1010 show?",
    "Explain Figure 8C-1"
]

for q in tests:
    text_results, page_results = fuse_results(q)
    print("=" * 100)
    print("QUERY:", q)
    print("TEXT TOP:")
    for r in text_results[:2]:
        print(f"  Section {r['section']} | {r['title']} | Page {r['page_num']} | score={r['score']:.4f}")
    print("PAGE TOP:")
    for p in page_results[:3]:
        print(f"  Page {p['page_num']} | Figures={p['figure_captions']} | score={p.get('score', 0):.4f}")

QUERY: What does MUTCD say about pedestrian hybrid beacons?
TEXT TOP:
  Section 4J.01 | Application of Pedestrian Hybrid Beacons | Page 768 | score=0.7441
  Section 4J.02 | Design of Pedestrian Hybrid Beacons | Page 769 | score=0.7084
PAGE TOP:
  Page 768 | Figures=['Figure 4J-1 for the length of the crosswalk', 'Figure 4J-2 for the length of the crosswalk', 'Figure 4J-1. Guidelines for the Installation of Pedestrian'] | score=0.7441
  Page 769 | Figures=['Figure 4J-3)', 'Figure 4J-2. Guidelines for the Installation of Pedestrian'] | score=0.7084
  Page 770 | Figures=['Figure 4J-3)', 'Figure 4J-3. Sequence for a Pedestrian Hybrid Beacon'] | score=0.6478
QUERY: Warning Signs and Plaques for Grade Crossings
TEXT TOP:
  Section 8B.11 | EXEMPT Grade Crossing Plaques (R15-3P and W10-1aP) | Page 1044 | score=0.7540
  Section 8B.24 | Next Crossing Plaques (W10-14P and W10-14aP) | Page 1048 | score=0.7205
PAGE TOP:
  Page 1043 | Figures=['Figure 8B-1) should be used', 'Figure 8B-4. Warning Sig

In [ ]:
# =========================
# FINAL MULTIMODAL QA CELL
# =========================

import re
import numpy as np
from PIL import Image
import torch

# -------------------------
# 1. QUERY UNDERSTANDING
# -------------------------
def detect_query_type(query: str):
    q = query.lower()

    visual_keywords = [
        "figure", "diagram", "layout", "placement", "example",
        "intersection", "roundabout", "circular", "typical application",
        "lane closure", "marking", "sign placement"
    ]

    if any(k in q for k in visual_keywords):
        return "VISUAL_SCENARIO"
    return "TEXTUAL"

def extract_query_components(query: str):
    q = query.lower()

    context_terms = []
    if "circular" in q or "roundabout" in q:
        context_terms.append("circular intersection")
    if "intersection" in q:
        context_terms.append("intersection")

    return {
        "raw": query,
        "context_terms": context_terms
    }

# -------------------------
# 2. HYBRID RETRIEVAL
# -------------------------
def hybrid_retrieve(query, top_k=5):
    query_vec = embed_model.encode([query], normalize_embeddings=True)[0]

    # ---- TEXT RETRIEVAL ----
    text_scores = np.dot(section_embeddings, query_vec)
    top_text_idx = np.argsort(text_scores)[::-1][:top_k]

    text_results = [section_chunks[i] for i in top_text_idx]

    # ---- IMAGE RETRIEVAL ----
    image_scores = np.dot(page_embeddings, query_vec)

    # Boost figure captions
    for i, rec in enumerate(page_records):
        caption_text = " ".join(rec.get("figure_captions", [])).lower()
        if any(word in caption_text for word in query.lower().split()):
            image_scores[i] += 0.3  # BOOST

    top_img_idx = np.argsort(image_scores)[::-1][:top_k]
    image_results = [page_records[i] for i in top_img_idx]

    return text_results, image_results

# -------------------------
# 3. CONTEXT FILTERING
# -------------------------
def filter_results(query_info, text_results, image_results):
    context_terms = query_info["context_terms"]

    def is_relevant(text):
        t = text.lower()
        return all(term in t for term in context_terms) if context_terms else True

    filtered_text = [r for r in text_results if is_relevant(r["text"])]
    filtered_images = [r for r in image_results if is_relevant(r["page_text_norm"])]

    # fallback if too strict
    if not filtered_text:
        filtered_text = text_results[:2]
    if not filtered_images:
        filtered_images = image_results[:3]

    return filtered_text[:3], filtered_images[:4]

# -------------------------
# 4. CLEAN CONTEXT
# -------------------------
def clean_context(text_results):
    seen = set()
    cleaned = []

    for r in text_results:
        txt = re.sub(r"(Standard|Option|Guidance):?\s*\d*", "", r["text"])
        txt = re.sub(r"\s+", " ", txt).strip()

        if txt[:200] not in seen:
            seen.add(txt[:200])
            cleaned.append(txt)

    return "\n\n".join(cleaned[:3])

# -------------------------
# 5. STRUCTURED PROMPT
# -------------------------
def build_prompt(question, context, query_type):
    base_prompt = f"""
You are an expert in MUTCD traffic control standards.

Answer the question using ONLY the context below.

DO NOT copy raw text.
DO NOT use 'Standard 01...' style.
Summarize clearly.

QUESTION:
{question}

CONTEXT:
{context}

Follow EXACT structure:

[1] Direct Answer:
(2–4 lines)

[2] Context:
Explain scenario

[3] Key Guidelines:
Bullet points

[4] Design / Operational Logic:
Explain why/how

"""

    if query_type == "VISUAL_SCENARIO":
        base_prompt += """
[5] Visual Interpretation:
Explain the diagram clearly

"""

    base_prompt += """
[6] References:
Sections + pages

Be precise. Avoid repetition.
"""
    return base_prompt

# -------------------------
# 6. MAIN ANSWER FUNCTION
# -------------------------
def answer_query_final(question):

    # Step 1: understand query
    query_type = detect_query_type(question)
    query_info = extract_query_components(question)

    # Step 2: retrieve
    text_results, image_results = hybrid_retrieve(question)

    # Step 3: filter
    text_results, image_results = filter_results(
        query_info, text_results, image_results
    )

    # Step 4: context
    context = clean_context(text_results)

    # Step 5: build prompt
    prompt = build_prompt(question, context, query_type)

    # Step 6: generate answer
    output = llm(
        prompt,
        max_new_tokens=400,
        do_sample=False
    )

    answer = output[0]["generated_text"]

    # -------------------------
    # 7. PREPARE IMAGES
    # -------------------------
    images = []
    for rec in image_results[:4]:
        try:
            img = Image.open(rec["image_path"])
            images.append(img)
        except:
            pass

    # -------------------------
    # 8. REFERENCES
    # -------------------------
    refs = []
    for r in text_results:
        refs.append(f"Section {r['section']} (Page {r['page_num']})")

    return {
        "answer": answer,
        "images": images,
        "references": refs
    }

# -------------------------
# 9. SIMPLE CHAT INTERFACE
# -------------------------
import gradio as gr

def chat_fn(message):
    result = answer_query_final(message)

    return (
        result["answer"],
        result["images"],
        "\n".join(result["references"])
    )

with gr.Blocks() as demo:
    gr.Markdown("## MUTCD Multimodal RAG (Improved)")

    inp = gr.Textbox(label="Ask a question")
    out_text = gr.Textbox(label="Answer")
    out_images = gr.Gallery(label="Relevant Figures")
    out_refs = gr.Textbox(label="References")

    btn = gr.Button("Submit")
    btn.click(chat_fn, inputs=inp, outputs=[out_text, out_images, out_refs])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71290f6828683347c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
def on_send(message, history):
    answer, gallery_items, refs = answer_question(message)
    final_text = answer
    if refs:
        final_text += f"\n\nTop references:\n{refs}"
    history = history + [(message, final_text)]
    return history, "", gallery_items, refs, history

def on_clear():
    return [], "", [], "", []

with gr.Blocks(title="MUTCD Multimodal RAG", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## MUTCD Multimodal RAG")
    gr.Markdown("Dual retrieval: text sections and full pages are retrieved independently, then fused before VLM answering.")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=520, label="Chat")
            message = gr.Textbox(
                label="Question",
                placeholder="Example: What does page 1010 show?"
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear")

        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Relevant MUTCD pages", height=580, preview=True)
            refs = gr.Textbox(label="References", lines=10)

    state = gr.State([])

    send_btn.click(
        fn=on_send,
        inputs=[message, state],
        outputs=[chatbot, message, gallery, refs, state]
    )

    message.submit(
        fn=on_send,
        inputs=[message, state],
        outputs=[chatbot, message, gallery, refs, state]
    )

    clear_btn.click(
        fn=on_clear,
        inputs=[],
        outputs=[chatbot, message, gallery, refs, state]
    )

demo.launch(share=True, inbrowser=True, debug=True)

/tmp/ipykernel_912/1880779710.py:13: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="MUTCD Multimodal RAG", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_912/1880779710.py:19: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=520, label="Chat")
/tmp/ipykernel_912/1880779710.py:19: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=520, label="Chat")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f8c3f55fd5a25d4cc5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
